In [2]:
import pandas as pd
from scipy import stats
from ucimlrepo import fetch_ucirepo

In [3]:
heart=fetch_ucirepo(id=45)


In [4]:
df=heart.data.features

In [5]:
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0


In [6]:
# Make the three columns easy to read (see the table above):
df = df.rename(columns={"thalach": "max_heart_rate"})        # new, clear column name


df["sex"] = df["sex"].map({1: "male", 0: "female"})          # map = swap codes for words
df["disease"] = (heart.data.targets["num"] > 0).map({True: "yes", False: "no"})
#   num 0 → "no" (no heart disease),  num 1 to 4 → "yes" (heart disease)

print(df[["sex", "max_heart_rate", "disease"]].head())   # look at the first 5 rows

      sex  max_heart_rate disease
0    male             150      no
1    male             108     yes
2    male             129     yes
3    male             187      no
4  female             172      no


 Question 1. Do patients with heart disease reach a different maximum heart rate during exercise? 
• H₀: patients with and without heart disease have the same average maximum heart rate.
• H₁: the averages are different.

In [7]:
#t-test

#Splitting the data into two groups, with and without disease
with_disease=df.loc[df["disease"]=="yes","max_heart_rate"]
without_disease=df.loc[df["disease"]=="no","max_heart_rate"]

In [8]:
#printing mean for each group
print(without_disease.mean(),with_disease.mean())
print(f"The difference is {without_disease.mean() - with_disease.mean()}")

158.3780487804878 139.25899280575538
The difference is 19.11905597473242


In [9]:
print(stats.ttest_ind(with_disease,without_disease).pvalue)

3.4569649084302086e-14


Interpretation:


p-value: far below 0.05, so you reject H₀. The difference is real, not chance.

Effect size: patients with heart disease reached about 19 beats per minute lower (139 vs 158).

Plain sentence for the director: "During exercise, people with heart disease have a heart rate 19 beats per minute slower than healthy people. This clear drop is a real warning sign".

Question 2. Is heart disease more common in men or in women?

• H₀: heart disease is equally common in men and women.
• H₁: it is more common in one of them.

In [10]:
#chi-square test

# crosstab counts how many patients have each pair of values:
# female + no, female + yes, male + no, male + yes. It builds the count table for you.
table = pd.crosstab(df["sex"], df["disease"])
print(table)    # show the count table

# Run the chi-square test on the count table. It gives the p-value:
print(stats.chi2_contingency(table).pvalue)

disease  no  yes
sex             
female   72   25
male     92  114
2.666712348180942e-06


### Interpretation: sex vs heart disease

From the count table:
- Women: 25 of 97 had heart disease = **26%**
- Men: 114 of 206 had heart disease = **55%**

The chi-square p-value is 2.7 × 10⁻⁶ (0.0000027), far below 0.05, so I reject H₀. The difference between men and women is real, not chance.

**In plain words:** in this group of patients, heart disease was about twice as common in men as in women.

**Caution:** this shows a link, not a cause. Other factors that differ between men and women (such as age or smoking) may explain part of it.


 How big is the heart-rate difference? (Cohen's d)

In [17]:
# .std() gives the standard deviation: how spread out the heart rates are in each group
sd_with = with_disease.std()
sd_without = without_disease.std()

In [18]:
# Combine the two spreads into one (square each, average them, take the square root)
combined_sd = ((sd_with**2 + sd_without**2) / 2) ** 0.5

In [19]:
# The gap between the two averages (about 19 bpm)
gap = without_disease.mean() - with_disease.mean()

In [20]:
# Cohen's d = gap ÷ spread
cohens_d = gap / combined_sd
print(round(cohens_d, 2))

0.91


 about 0.2 = small, 0.5 = medium, 0.8 or more = large

 so with 0.91, the difference is big!

- **Effect size:** Cohen's d = 0.91, which is a large difference. The 19 bpm gap is not small: it really matters.


 Could age explain the heart-rate result? (a confounder check)

In [21]:
# & means "and": both conditions must be true for the row to be kept

# Younger patients (under 55)
young_with    = df.loc[(df["age"] < 55) & (df["disease"] == "yes"), "max_heart_rate"]
young_without = df.loc[(df["age"] < 55) & (df["disease"] == "no"),  "max_heart_rate"]

# Older patients (55 and over)
old_with      = df.loc[(df["age"] >= 55) & (df["disease"] == "yes"), "max_heart_rate"]
old_without   = df.loc[(df["age"] >= 55) & (df["disease"] == "no"),  "max_heart_rate"]

print("Under 55:   ", young_with.mean(), "vs", young_without.mean())
print("55 and over:", old_with.mean(), "vs", old_without.mean())

Under 55:    142.8181818181818 vs 164.07070707070707
55 and over: 137.61052631578949 vs 149.7076923076923


- **Age check:** the heart-rate gap remained after splitting by age: 21 bpm lower in patients under 55, and 12 bpm lower in patients 55 and over. So age does not explain the result, although older patients have lower heart rates in general.
